In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Reading the dataframe

In [ ]:
df_yield = pd.read_csv("/kaggle/input/crop-yield-prediction-dataset/yield.csv")
df_temp= pd.read_csv("/kaggle/input/crop-yield-prediction-dataset/temp.csv")
df_rainfall = pd.read_csv("/kaggle/input/crop-yield-prediction-dataset/rainfall.csv")
df_pesticides = pd.read_csv("/kaggle/input/crop-yield-prediction-dataset/pesticides.csv")
df_yield_df = pd.read_csv("/kaggle/input/crop-yield-prediction-dataset/yield_df.csv")

# Checking the dataframes and looking into the common columns or columns of relevance

In [ ]:
#### Checking for a column to implement the join 

print(df_yield.columns)
print(df_yield['Area'].nunique())
print(df_yield['Year'].nunique())

In [ ]:

print(df_yield_df.columns)
print(df_yield_df['Area'].nunique())
print(df_yield_df['Year'].nunique())
df_yield_df.drop(['Unnamed: 0'], axis = 1 ,inplace = True)

In [ ]:
print(df_pesticides.columns)
print(df_pesticides['Area'].nunique())
print(df_pesticides['Year'].nunique())

In [ ]:
print(df_rainfall.columns)
## Renaming the ' Area' column
df_rainfall.rename(columns = {' Area':'Area'},inplace = True)
print(df_rainfall['Area'].nunique())
print(df_rainfall['Year'].nunique())

In [ ]:
print(df_temp.columns)
df_temp.rename(columns = {'year':'Year','country':'Area'},inplace = True)
print(df_temp['Area'].nunique())
print(df_temp['Year'].nunique())
print(df_temp.columns)

#### The common columns in the above are the Year and and Country(Area) column , we will merge our datasets on these columns

In [ ]:
df_temprain = pd.merge(df_rainfall,df_temp,on = ['Year','Area'])
df_trp = pd.merge(df_temprain,df_pesticides , on = ['Year','Area'])

In [ ]:
trpl =list(df_trp.columns)
dfy = list(df_yield.columns)
com1 = [i for i in trpl if i in dfy]

In [ ]:
print(df_yield.columns)
print(df_yield_df.columns)



ly = [i for i in list(df_yield.columns) if i in (df_yield_df.columns)]

In [ ]:
yield_df = pd.merge(df_yield_df,df_yield, on = ['Year','Area','Item'])

In [ ]:
print(yield_df.shape)

print(yield_df.columns)
print(df_trp.columns)

In [ ]:
# Assuming your first dataframe is df1 and second dataframe is df2
years_df1 = set(yield_df['Year'].unique())
years_df2 = set(df_trp['Year'].unique())

# Find the years in df2 that are not in df1
years_only_in_df2 = years_df2 - years_df1

# Print the result
print("Years present in df1 but not in df2:", years_only_in_df2)

In [ ]:
# Assuming your first dataframe is df1 and second dataframe is df2
area_df1 = set(yield_df['Year'].unique())
area_df2 = set(df_trp['Year'].unique())

# Find the years in df2 that are not in df1
area_only_in_df1 = area_df2 - area_df1

# Print the result
print("Area present in df1 but not in df2:", area_only_in_df1)

#### We see that the columns in the respective datasets are the same and the values of Year and Area overlap. Thus we have repetitive data , and hence ignoring the adf_trp dataset .Proceeding with the yield_df.



# Exploratory data analysis

In [ ]:
yield_df.head()

In [ ]:
print(yield_df['Item Code'].nunique())
print(yield_df['Item'].nunique())
## It has one unique value for each crop

In [ ]:
print(yield_df['Domain'].nunique())
print(yield_df['Domain Code'].nunique())

In [ ]:
print(yield_df['Area'].nunique())
print(yield_df['Area Code'].nunique())

In [ ]:
print(yield_df['Element Code'].nunique())
print(yield_df['Element'].nunique())
print(yield_df['Unit'].nunique())

#### From the above it is evident that the Year Code and the Domain code are duplicate and irrelevent columns respectively. The Area code column is not useful for us for drawing any tangible conclusion. Domain also has only one value and hence dropping it. Dropping the columns. Area code is also representative of the Area code. hg/ha_yield And Value and Unit are all connected and hence dropping the last 2 columns .

In [ ]:
## Making a copy of the dataset , because it may be required for later use

Yield_final_data = yield_df.copy()

In [ ]:
Yield_final_data.drop(['Area Code','Year Code','Domain','Domain Code','Area Code','Item Code','Element','Element Code','Unit','Value'],axis = 1 , inplace = True)

In [ ]:
Yield_final_data.head()

In [ ]:
#### Filtering the dataset based on Area being India

Yield_final_data[Yield_final_data['Area'] == 'India'].head()



### The dataset must have the avg_temp in Celsius based on the temparature 

#### There is data in the Item column where there are multiple crops listed by ' ,'.

In [ ]:
Yield_final_data['Item'] = Yield_final_data['Item'].str.split(', ')
Yield_final_data = Yield_final_data.explode('Item').reset_index(drop=True)

In [ ]:
Yield_final_data.shape

In [ ]:
Yield_final_data.loc[Yield_final_data['Area'].str.len().sort_values().index].head(12)

In [ ]:
Yield_final_data['Item'].value_counts()

In [ ]:
Yield_final_data.info()


In [ ]:
#### The dataset columns seem to be in their correct type

In [ ]:
Yield_final_data.describe()

In [ ]:
Yield_final_data.groupby(['Area'],sort = True)[['hg/ha_yield']].sum().nlargest(10, 'hg/ha_yield')

In [ ]:
Yield_final_data.groupby(['Area','Item'], sort=True)['hg/ha_yield'].sum().nlargest(10)

In [ ]:
Yield_final_data.groupby(['Item','Area'], sort=True)['hg/ha_yield'].sum().nlargest(10)

#### The datasets show that India is the highest datapoint in the Area column and 'Potato' is the dominant crop 

In [ ]:
## Checking the pairplot between the columns
import seaborn as sns
sns.pairplot(Yield_final_data)

In [ ]:
# Checking the data for outliers

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize = (10,10))

plt.subplot(3,2,1)
sns.boxplot(data= Yield_final_data['Year'])
plt.title('Year')


plt.subplot(3,2,2)
sns.boxplot(data= Yield_final_data['hg/ha_yield'])
plt.title('hg/ha_yield')

plt.subplot(3,2,3)
sns.boxplot(data= Yield_final_data['average_rain_fall_mm_per_year'])
plt.title('average_rain_fall_mm_per_year')

plt.subplot(3,2,4)
sns.boxplot(data= Yield_final_data['pesticides_tonnes'])
plt.title('pesticides_tonnes')

plt.subplot(3,2,5)
sns.boxplot(data= Yield_final_data['avg_temp'])
plt.title('avg_temp')

plt.show()

#### We see that there are certain outliers in the avg_temp and pesticides_tonnes columns. But we will not remove anything as of yet.

In [ ]:
Yield_final_data.info()

In [ ]:
Yield_final_data['Item'].value_counts()

In [ ]:
### Paddy and Rice are the same crop and hence renaming them
##Plantains and others may be any number of distinct crops in over 100 countries so dropping them because they may not anything valuable

Yield_final_data['Item'] = Yield_final_data['Item'].str.replace('paddy','Rice')
Yield_final_data = Yield_final_data[Yield_final_data['Item'] != 'Plantains and others']
Yield_final_data['Item'].value_counts()

In [ ]:
num_cor = Yield_final_data.select_dtypes(['int64','float64']).corr()

In [ ]:
sns.heatmap(num_cor,cmap = 'YlGnBu',annot = True)
plt.title('Heatmap')


#### The plot shows that there is no deep correlation between any columns

In [ ]:
## Checking the data distribution in the 
sns.histplot(Yield_final_data, x = 'Year' , bins = range(1985, 2020, 5))


In [ ]:
## Checking the data distribution in the yield column
sns.histplot(Yield_final_data, x = 'hg/ha_yield' )

In [ ]:
### Checking the pesticide usage data

## Checking the data distribution in the yield column
sns.histplot(Yield_final_data, x = 'pesticides_tonnes' )

There are outlier points in the datsets and hence dropping only the upper 10% data points to prevent high variance. 

In [ ]:
Yield_final_data = Yield_final_data[Yield_final_data['pesticides_tonnes'] <= Yield_final_data['pesticides_tonnes'].quantile(0.90)]

In [ ]:
## Checking the data distribution in the yield column(if there have been any changes in the distribution after keeping only 90th quantile of yield)
sns.histplot(Yield_final_data, x = 'hg/ha_yield' )

In [ ]:
### There doesnt seem to be much change and hence dropping the upper 5th quantile

Yield_final_data = Yield_final_data[Yield_final_data['hg/ha_yield'] <= Yield_final_data['hg/ha_yield'].quantile(0.95)]

In [ ]:
sns.histplot(Yield_final_data, x = 'hg/ha_yield' )

In [ ]:
sns.histplot(Yield_final_data, x = 'pesticides_tonnes' )

In [ ]:
num_cor = Yield_final_data.select_dtypes(['int64','float64']).corr()
sns.heatmap(num_cor,cmap = 'YlGnBu',annot = True)
plt.title('Heatmap')

#### The features still do not bear high correlation and even though the pesticide and yield columns bear left skewness , we will not drop any rows any further fearing loss of information.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_squared_error , mean_absolute_error , r2_score



In [ ]:
Yield_final_data = pd.get_dummies(Yield_final_data,columns = ['Item','Area'], drop_first = True)

# Splitting the data in train ,test split 

In [ ]:
### As of 

split_ratio = 0.25  
split_index = int(len(Yield_final_data) * split_ratio)

# Get the 'Year' value at the split point
split_year = Yield_final_data['Year'].iloc[split_index]

print(f"Split Year: {split_year}")

In [ ]:
df_train = Yield_final_data[Yield_final_data['Year'] <= 2009]
df_test  = Yield_final_data[Yield_final_data['Year'] > 2009]



In [ ]:
Yield_final_data.shape

In [ ]:
df_train = df_train.drop('Year',axis = 1)
df_test  = df_test.drop('Year',axis = 1)

X_train = df_train.drop('hg/ha_yield',axis = 1 )
y_train = df_train['hg/ha_yield']
X_test  = df_test.drop('hg/ha_yield', axis = 1)
y_test  = df_test['hg/ha_yield']

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

In [ ]:
y_train.values.reshape(-1,1)
y_test.values.reshape(-1,1)

# Scaling and Model training

In [ ]:


mmc = MinMaxScaler()
X_train = mmc.fit_transform(X_train)
X_test = mmc.transform(X_test)

In [ ]:
### training the model with LR



from sklearn.linear_model import LinearRegression
LR = LinearRegression()
LR.fit(X_train,y_train)
y_pred = LR.predict(X_test)

Model_perf = pd.DataFrame(columns=['Model_Name','MSE','R2_Score'])

LR_mse = mean_squared_error(y_test,y_pred) 
LR_R2 = r2_score(y_test,y_pred)

new_row = {'Model_Name':'Linear Regression','MSE':LR_mse , 'R2_Score': LR_R2}
Model_perf = Model_perf.append(new_row,ignore_index = True)

In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures


# Create polynomial features
poly_features = PolynomialFeatures(degree=2)  
X_poly = poly_features.fit_transform(X_train)

# Train the polynomial regression model
poly_regression = LinearRegression()
poly_regression.fit(X_poly, y_train)

# Predict using the trained model

X_test_poly = poly_features.transform(X_test)
y_pred = poly_regression.predict(X_test_poly)
print("Predicted values:", y_pred)




In [ ]:
PR_mse = mean_squared_error(y_test,y_pred) 
PR_R2 = r2_score(y_test,y_pred)

new_row = {'Model_Name':'Polynomial Regression(degree 2)','MSE':PR_mse , 'R2_Score': PR_R2}
Model_perf = Model_perf.append(new_row,ignore_index = True)

In [ ]:
### The polynomial regression does not perform that well

In [ ]:
### Trying the XGB regressor now

from xgboost import XGBRegressor
XG_boost = XGBRegressor(max_depth = 3,n_estimators = 300 )

XG_boost.fit(X_train , y_train)
y_pred = XG_boost.predict(X_test)
XG_rmse = mean_squared_error(y_test,y_pred)
XG_R2 = r2_score(y_test,y_pred)

new_row = {'Model_Name':'XGB','MSE':XG_rmse ,'R2_Score': XG_R2}
Model_perf = Model_perf.append(new_row,ignore_index = True)

# Model_Performance

In [ ]:
Model_perf